<a href="https://colab.research.google.com/github/nitin04-stack/Text-Summarizer-using-T5-Transformer/blob/main/text_summerizer(NLP)using_trnasformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer,TrainingArguments,T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/samsum-train.csv")
val_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/samsum-validation.csv")

In [ ]:
train_data

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
train_data = train_data.sample(n=4000 , random_state = 42).reset_index(drop=True)
val_data = val_data.sample(n=500,random_state = 42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

In [ ]:
val_data.shape

(500, 3)

In [ ]:
val_data

,id,dialogue,summary
0,13680857,"Edd: wow, did you hear that they're transferri...",Rose and Edd will be transferred to a new depa...
1,13716124,"Tom: Where is the ""Sala del Capitolo""\r\nKevin...","""Sala del Capitolo"" Tom is looking for is in t..."
2,13864418,Patricia: The rowing practice is cancelled!\nK...,The rowing practice is cancelled. A few member...
3,13729340,"Tom: U OK?\r\nAlex: Yeah, pretty good. U?\r\nT...",Tom and Alex had fun last night. They drank a ...
4,13818813,"Patricia: Hello, here's the fair-trade brand I...",Patricia recommends a fair-trade brand she tal...
...,...,...,...
495,13820618,Izzy: Anyone knows where professor Xavier has ...,Professor Xavier probably holds his duty hours...
496,13680766,"Rita: didn't take breakfast with me, is there ...",Lina will give Rita one of her 2 sandwiches.
497,13819081,"Peter: Yo, we’re coming over to Warsaw for thi...",Peter and Jen are coming to Warsaw for the wee...
498,13862819,"Francis: Hey\nFrancis: Listen, I need a favor....",Francis asked Reynold for help with installing...


In [ ]:
#  data preprocessing
import re

In [ ]:
def clean_text(text):
  text = re.sub(r"\r\n"," ",text)
  text = re.sub(r"\s+"," ",text)
  text = re.sub(r"<.*?>"," ",text)
  text = text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_text)
train_data["summary"] = train_data["summary"].apply(clean_text)

val_data["dialogue"] = val_data["dialogue"].apply(clean_text)
val_data["summary"] = val_data["summary"].apply(clean_text)


In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [ ]:
# tokernization meqns covert text into numbers

In [ ]:
tokenzier = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def tokanize(data):
  inputs = tokenzier(data["dialogue"] , padding = "max_length" , max_length = 512,truncation=True)
  target = tokenzier(data["summary"] , padding = "max_length" , max_length = 150,truncation=True)

  inputs["labels"] = target["input_ids"]
  return inputs

In [ ]:
train_dataset = train_data.apply(tokanize,axis = 1).tolist()
val_dataset = val_data.apply(tokanize,axis = 1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# for training
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy="epoch",
    save_strategy = "epoch",
    warmup_steps= 500

)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.575382,0.380237
2,0.397010,0.360123
3,0.374551,0.355223
4,0.362248,0.350841
5,0.356006,0.349393
6,0.351044,0.349150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9027068430582682, metrics={'train_runtime': 1298.3773, 'train_samples_per_second': 18.485, 'train_steps_per_second': 2.311, 'total_flos': 3248203235328000.0, 'train_loss': 0.9027068430582682, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenzier.save_pretrained("./saved_summary_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [ ]:
# if we want to use this save trained model summary
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenzier = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
#  test the core logic of summerization

In [ ]:
def summarize_dailogue(dialogue):
  dialogue = clean_text(dialogue)

  # tokenize
  inputs = tokenzier(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  ).to(device)
  # generate summary in the form of tokens
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length =150,
      num_beams = 4, # to polish out outputs
      early_stopping = True
  )
  # token_ids convert to summary
  summary = tokenzier.decode(targets[0],skip_special_tokens = True)
  return summary


In [ ]:
test_dialogue = """Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers,
and industry leaders will be crucial to ensure that AI systems are developed
and used in a safe and beneficial way. """

summarize_dailogue(test_dialogue)

'ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security and long-term societal impact.'

In [ ]:
!cp -r /content/savade_summary_model/content/drive/MyDrive/Colab Notebooks/saved_summary_model/ /content/drive/MyDrive/Colab Notebooks/saved_summary_model/

cp: target 'Notebooks/saved_summary_model/' is not a directory


In [ ]:
!zip -r saved_model.zip /content/results

  adding: content/results/ (stored 0%)
  adding: content/results/checkpoint-1000/ (stored 0%)
  adding: content/results/checkpoint-1000/training_args.bin (deflated 53%)
  adding: content/results/checkpoint-1000/optimizer.pt (deflated 7%)
  adding: content/results/checkpoint-1000/generation_config.json (deflated 29%)
  adding: content/results/checkpoint-1000/trainer_state.json (deflated 63%)
  adding: content/results/checkpoint-1000/rng_state.pth (deflated 26%)
  adding: content/results/checkpoint-1000/config.json (deflated 63%)
  adding: content/results/checkpoint-1000/scheduler.pt (deflated 61%)
  adding: content/results/checkpoint-1000/model.safetensors (deflated 10%)
  adding: content/results/checkpoint-2500/ (stored 0%)
  adding: content/results/checkpoint-2500/training_args.bin (deflated 53%)
  adding: content/results/checkpoint-2500/optimizer.pt (deflated 7%)
  adding: content/results/checkpoint-2500/generation_config.json (deflated 29%)
  adding: content/results/checkpoint-2500/

In [1]:
model.save_pretrained("/content/drive/MyDrive/ColabNotebooks/saved_summary_model")
tokenzier.save_pretrained("/content/drive/MyDrive/ColabNotebooks/saved_summary_model")

NameError: name 'model' is not defined